# 09 · Multi-token prediction (MTP), DeepSeek-V3 style

**Needed for:** DeepSeek-V3 (LLM track, step 7). The same modules can double as a draft model for speculative
decoding.

A normal language model predicts only the next token $t_{i+1}$ from position $i$. MTP adds $D$ small sequential
modules. Module $k$ predicts $t_{i+k+1}$, combining:
- the previous depth's hidden state at position $i$ ($h^{k-1}_i$, where depth 0 is the main model), and
- the embedding of the token $k$ steps ahead, $t_{i+k}$, which is known during training.

$$h'^{k}_i = M_k\big[\text{RMSNorm}(h^{k-1}_i);\ \text{RMSNorm}(\text{Emb}(t_{i+k}))\big], \qquad h^k = \text{Block}_k(h'^k)$$

Embedding and output head are **shared** with the main model. The extra losses densify the training signal.
Getting the index alignment right is the whole difficulty, so that's what the tests check.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. A tiny causal trunk and the MTP modules

In [ ]:
def causal_block(d, heads=4):
    return nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0, batch_first=True, norm_first=True)


def run_causal(block, h):
    T = h.shape[1]
    mask = nn.Transformer.generate_square_subsequent_mask(T, device=h.device)
    return block(h, src_mask=mask, is_causal=True)


class MTPModel(nn.Module):
    def __init__(self, vocab, d=64, n_layers=2, depth=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.trunk = nn.ModuleList(causal_block(d) for _ in range(n_layers))
        self.norm = nn.RMSNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)                   # shared by all depths
        self.mtp_norm_h = nn.ModuleList(nn.RMSNorm(d) for _ in range(depth))
        self.mtp_norm_e = nn.ModuleList(nn.RMSNorm(d) for _ in range(depth))
        self.mtp_proj = nn.ModuleList(nn.Linear(2 * d, d, bias=False) for _ in range(depth))
        self.mtp_block = nn.ModuleList(causal_block(d) for _ in range(depth))
        self.depth = depth


def mtp_forward(model, tokens):
    """tokens: (B, T). Returns a list of (logits, targets): index 0 is the main next-token head,
    index k is MTP depth k. logits: (B, L, V), targets: (B, L)."""
    h = model.emb(tokens)
    for blk in model.trunk:
        h = run_causal(blk, h)
    outputs = [(model.head(model.norm(h[:, :-1])), tokens[:, 1:])]
    T = tokens.shape[1]
    for k in range(1, model.depth + 1):
        emb_ahead = model.emb(tokens[:, k:])                    # Emb(t_{i+k}) for i = 0 .. T-k-1
        h = h[:, :T - k]                                        # h^{k-1}_i for the same i
        h = model.mtp_proj[k - 1](torch.cat([model.mtp_norm_h[k - 1](h), model.mtp_norm_e[k - 1](emb_ahead)], -1))
        h = run_causal(model.mtp_block[k - 1], h)
        outputs.append((model.head(model.norm(h[:, :-1])), tokens[:, k + 1:]))  # predicts t_{i+k+1}
    return outputs

## 2. Test the alignment with causality

At depth $k$, the prediction at position $i$ may see tokens $t_0 \dots t_{i+k}$ and nothing later. So changing
token $p$ must leave depth-$k$ logits unchanged at every position $i < p - k$, and must change position $p - k$.
Off-by-one errors break this test in one direction or the other.

In [ ]:
def test_mtp_alignment(fn):
    torch.manual_seed(0)
    model = MTPModel(vocab=50, depth=2).eval()
    B, T = 2, 12
    tokens = torch.randint(0, 50, (B, T))
    outs = fn(model, tokens)
    for k, (logits, targets) in enumerate(outs):
        assert logits.shape[:2] == targets.shape == (B, T - k - 1), (k, logits.shape, targets.shape)
        torch.testing.assert_close(targets, tokens[:, k + 1:])
    p = 7
    changed = tokens.clone()
    changed[:, p] = (changed[:, p] + 1) % 50
    outs2 = fn(model, changed)
    for k in range(len(outs)):
        a, b = outs[k][0], outs2[k][0]
        torch.testing.assert_close(a[:, :p - k], b[:, :p - k])             # before p-k: unaffected
        assert not torch.allclose(a[:, p - k], b[:, p - k]), f"depth {k}: position {p - k} must see token {p}"


test_mtp_alignment(mtp_forward)
print("alignment and causality ok")

## 3. Training with the MTP loss

$$\mathcal{L} = \mathcal{L}_{main} + \frac{\lambda}{D}\sum_{k=1}^{D}\mathcal{L}_{MTP}^k$$

DeepSeek-V3 uses $D = 1$, with $\lambda = 0.3$ early in training and 0.1 later. Toy data below: repeated random
patterns, so tokens several steps ahead are predictable once the model has seen part of the pattern.

In [ ]:
def make_batch(B=32, T=32, vocab=50, period=6):
    base = torch.randint(0, vocab, (B, period))
    return base.repeat(1, T // period + 1)[:, :T]


model = MTPModel(vocab=50, depth=2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
lam = 0.3
for step in range(301):
    tokens = make_batch().to(device)
    outs = mtp_forward(model, tokens)
    losses = [F.cross_entropy(lg.reshape(-1, lg.shape[-1]), tg.reshape(-1)) for lg, tg in outs]
    loss = losses[0] + lam * sum(losses[1:]) / model.depth
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 100 == 0:
        print(f"step {step:3d}  main {losses[0].item():.3f}  " + "  ".join(f"depth{k} {l.item():.3f}" for k, l in enumerate(losses[1:], 1)))

The MTP losses end *below* the main loss here. That's not a bug: depth $k$ also sees the real token $t_{i+k}$,
one more known token of the pattern. On this toy data the first period of every sequence is random, so that
extra token helps a lot.

## 4. At inference

- Simplest: drop the MTP modules and use the main model as usual.
- **Speculative decoding:** the depth-1 module drafts $t_{i+2}$ cheaply. The next main-model forward pass checks
  it; if the main model agrees, you accept two tokens for one step. DeepSeek reports 85–90% acceptance for the
  second token, about 1.8× tokens per second.

## Things to be able to explain

- Why sequential modules (each depth sees the previous depth's state) rather than independent parallel heads
  (Gloeckle et al. 2024)? (Keeps the causal chain for each prediction.)
- Why does MTP help the *main* model? (Denser signal; the trunk must plan beyond one token.)

## Drill

In [ ]:
def my_mtp_forward(model, tokens):
    """Use the MTPModel attributes above; return the same list of (logits, targets)."""
    raise NotImplementedError


drill(test_mtp_alignment, my_mtp_forward)